# Viettel AI Race - Qwen 2.5 7B Pipeline
Pipeline này được tối ưu để chạy trên **Kaggle GPU (T4 x2 hoặc P100)**.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes requests tqdm

## Hướng dẫn kết nối Dữ Liệu
1. Ở cột bên phải Kaggle, nhấn **Add Data** -> Upload file `input.zip` của bạn lên.
2. Sau khi upload, đường dẫn sẽ thường là `/kaggle/input/[tên-dataset]/input`. Hãy sửa biến `INPUT_DIR` bên dưới cho phù hợp.

In [ ]:
INPUT_DIR = "/kaggle/input/viettel-ai-data/input" # <-- SỬA ĐƯỜNG DẪN NÀY NẾU CẦN
OUTPUT_DIR = "/kaggle/working/output"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import json
import re
import requests
from tqdm import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

def clean_drug_name(drug_string):
    s = drug_string.lower()
    patterns = [
        r'\b\d+(\.\d+)?\s*(mg|g|mcg|ml|l|units|iu)\b',
        r'\b(po|iv|im|sc|sl|pr)\b',
        r'\b(daily|bid|tid|qid|q\d+h|prn|qam|qhs)\b',
        r'\b(tablet|capsule|suspension|oral|injection|syrup)\b',
        r'điều trị.*',
        r'[:\-]'
    ]
    for p in patterns:
        s = re.sub(p, ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s if s else drug_string.lower()

def get_rxnorm_cui_robust(drug_string):
    def query(name):
        url = f"https://rxnav.nlm.nih.gov/REST/rxcui.json?name={requests.utils.quote(name)}&search=2"
        try:
            resp = requests.get(url, timeout=5)
            if resp.status_code == 200:
                data = resp.json()
                if "idGroup" in data and "rxnormId" in data["idGroup"]:
                    return data["idGroup"]["rxnormId"][0]
        except:
            pass
        return None
        
    cui = query(drug_string)
    if cui: return cui
    
    cleaned = clean_drug_name(drug_string)
    if cleaned != drug_string.lower():
        cui = query(cleaned)
        if cui: return cui
        
    words = cleaned.split()
    if len(words) > 1:
        cui = query(words[0])
        if cui: return cui
    return None

print("Loading LLM...")
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quantization_config, device_map="auto")

def generate_extraction(text):
    system_prompt = """Bạn là một chuyên gia y tế trích xuất thực thể từ hồ sơ bệnh án tiếng Việt.
Bạn cần trích xuất chính xác các thực thể thuộc 3 loại: THUỐC, TRIỆU_CHỨNG, CHẨN_ĐOÁN.
Đồng thời xác định xem thực thể đó có nằm trong bệnh sử/tiền sử không (isHistorical).

Yêu cầu BẮT BUỘC:
- Trích xuất CHÍNH XÁC chuỗi con (substring) xuất hiện trong văn bản. Đừng thay đổi dấu câu hay viết hoa.
- Chỉ trả về ĐÚNG ĐỊNH DẠNG JSON MẢNG (List of Objects), không kèm theo văn bản giải thích nào khác.
- Các thuộc tính bắt buộc: "text", "type", "assertions".
- "assertions": Nếu thực thể nằm trong bệnh sử/tiền sử, gán ["isHistorical"]. Nếu là hiện tại, gán []."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Văn bản:\n{text}"}
    ]
    
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text_input], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(**model_inputs, max_new_tokens=2048, temperature=0.01, do_sample=False)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    match = re.search(r'\[.*\]', response, re.DOTALL)
    if match:
        try: return json.loads(match.group(0))
        except: pass
    return []

def dump_strict_json(data, out_path):
    lines = ["[\n"]
    for i, item in enumerate(data):
        lines.append("  {\n")
        lines.append(f'    "text": {json.dumps(item["text"], ensure_ascii=False)},\n')
        lines.append(f'    "type": {json.dumps(item["type"], ensure_ascii=False)},\n')
        if item.get("type") == "THUỐC" and "candidates" in item:
            lines.append(f'    "candidates": {json.dumps(item["candidates"], ensure_ascii=False)},\n')
        lines.append(f'    "assertions": {json.dumps(item.get("assertions", []), ensure_ascii=False)},\n')
        lines.append(f'    "position": {json.dumps(item["position"])}\n')
        lines.append("  }\n" if i == len(data) - 1 else "  },\n")
    lines.append("]")
    with open(out_path, 'w', encoding='utf-8') as out_f:
        out_f.writelines(lines)

files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.txt')]
for fname in tqdm(files, desc="Processing"):
    in_path = os.path.join(INPUT_DIR, fname)
    out_path = os.path.join(OUTPUT_DIR, fname.replace('.txt', '.json'))
    
    with open(in_path, 'r', encoding='utf-8') as f:
        text = f.read()
        
    entities = generate_extraction(text)
    formatted = []
    for item in entities:
        ext_text = item.get("text", "")
        start_idx = text.find(ext_text)
        if start_idx == -1: continue
        
        etype = item.get("type", "TRIỆU_CHỨNG")
        if etype not in ["THUỐC", "TRIỆU_CHỨNG", "CHẨN_ĐOÁN"]: etype = "TRIỆU_CHỨNG"
            
        new_item = {"text": ext_text, "type": etype}
        if etype == "THUỐC":
            cui = get_rxnorm_cui_robust(ext_text)
            if cui: new_item["candidates"] = [cui]
                
        asts = item.get("assertions", [])
        new_item["assertions"] = ["isHistorical"] if "isHistorical" in asts else []
        new_item["position"] = [start_idx, start_idx + len(ext_text)]
        formatted.append(new_item)
        
    dump_strict_json(formatted, out_path)
print("Inference Complete!")

In [ ]:
!cd /kaggle/working/output && zip -q -r /kaggle/working/output_root.zip *
print("Đã nén xong! Download output_root.zip ở panel Output bên phải Kaggle nhé.")